# Signature Pattern Discovery (Data-Driven)

Discover anomaly patterns **directly from training data** — no hand-crafted guesses.

**Approach**:
1. Extract operation fingerprints from every anomaly session
2. Cluster sessions by fingerprint similarity
3. Name each cluster by its dominant operations
4. Generate `HDFS_ERROR_PATTERNS` and `BGL_ERROR_PATTERNS` from real data

Start with **HDFS**, then **BGL**.

## 1. Setup

In [1]:
import sys
import re
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import HDFSDataLoader, BGLDataLoader, Session

print('Imports OK')

Imports OK


## 2. Load HDFS Training Data

In [2]:
loader = HDFSDataLoader(
    log_file='../logs/HDFS.log',
    label_file='../logs/anomaly_label_HDFS.csv'
)
loader.load()

train_sessions = loader.get_train()
hdfs_anomalies = [s for s in train_sessions if s.label == 1]
hdfs_normals   = [s for s in train_sessions if s.label == 0]

print(f'Training sessions: {len(train_sessions):,}')
print(f'  Anomalies: {len(hdfs_anomalies):,}')
print(f'  Normals:   {len(hdfs_normals):,}')
lengths = [len(s.lines) for s in hdfs_anomalies]
print(f'\nAnomaly line-length: min={min(lengths)}, max={max(lengths)}, median={sorted(lengths)[len(lengths)//2]}')

Loading HDFS logs from: ../logs/HDFS.log
Loading labels from: ../logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:10, 1033644.04it/s]


Found 575061 unique blocks
Training sessions: 402,542
  Anomalies: 11,786
  Normals:   390,756

Anomaly line-length: min=2, max=284, median=20


## 3. Automatic Token Discovery (HDFS)

Instead of hand-picking operation keywords, **discover them from the data**:
1. Extract all words from every session (anomaly + normal sample)
2. Compute per-session presence rate for anomaly vs normal
3. Keep only tokens where the rates differ significantly → **discriminative tokens**
4. These become the fingerprint features

In [3]:
import random
random.seed(42)
norm_sample = random.sample(hdfs_normals, min(len(hdfs_anomalies), len(hdfs_normals)))

# Step 1: Extract all words from anomaly and normal sessions
anom_word_presence = Counter()   # how many anomaly sessions contain this word
norm_word_presence = Counter()   # how many normal sessions contain this word

for s in hdfs_anomalies:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'[a-z]{3,}', text))
    for w in words:
        anom_word_presence[w] += 1

for s in norm_sample:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'[a-z]{3,}', text))
    for w in words:
        norm_word_presence[w] += 1

# Step 2: Compute discriminative score for each word
all_words = set(anom_word_presence.keys()) | set(norm_word_presence.keys())
disc_hdfs = []
for w in all_words:
    a_count = anom_word_presence.get(w, 0)
    n_count = norm_word_presence.get(w, 0)
    if a_count < 5 and n_count < 5:
        continue  # skip rare words
    a_rate = a_count / len(hdfs_anomalies)
    n_rate = n_count / len(norm_sample)
    # Discriminative = large absolute difference in rates
    disc = abs(a_rate - n_rate)
    direction = 'ANOM' if a_rate > n_rate else 'NORM'
    disc_hdfs.append((w, a_count, n_count, a_rate, n_rate, disc, direction))

disc_hdfs.sort(key=lambda x: -x[5])

# Step 3: Show top discriminative tokens
print(f'Top discriminative tokens (anomaly vs normal):')
print(f'{"Token":25s} {"Anom sess":>10s} {"Norm sess":>10s} {"Anom%":>7s} {"Norm%":>7s} {"Diff":>7s} {"Dir":>5s}')
print('-' * 75)
for w, ac, nc, ar, nr, disc, d in disc_hdfs[:50]:
    print(f'{w:25s} {ac:10d} {nc:10d} {ar:7.1%} {nr:7.1%} {disc:7.1%} {d:>5s}')

# Step 4: Auto-select discriminative features
# Keep tokens that are significantly more present in anomalies OR normals
# Threshold: |anom_rate - norm_rate| > 0.05 (5% difference)
DISC_THRESHOLD = 0.05
hdfs_features = [(w, ar, nr, d) for w, ac, nc, ar, nr, disc, d in disc_hdfs if disc > DISC_THRESHOLD]

print(f'\n=== Selected {len(hdfs_features)} discriminative features (threshold={DISC_THRESHOLD}) ===')
for w, ar, nr, d in hdfs_features:
    print(f'  {w:25s}  anom={ar:.1%}  norm={nr:.1%}  [{d}]')

Top discriminative tokens (anomaly vs normal):
Token                      Anom sess  Norm sess   Anom%   Norm%    Diff   Dir
---------------------------------------------------------------------------
not                             6700          5   56.8%    0.0%   56.8%  ANOM
blockmap                        7408      11786   62.9%  100.0%   37.1%  NORM
addstoredblock                  7408      11786   62.9%  100.0%   37.1%  NORM
size                            7408      11786   62.9%  100.0%   37.1%  NORM
updated                         7408      11786   62.9%  100.0%   37.1%  NORM
added                           7408      11786   62.9%  100.0%   37.1%  NORM
packetresponder                 7430      11786   63.0%  100.0%   37.0%  NORM
for                             7430      11786   63.0%  100.0%   37.0%  NORM
terminating                     7430      11786   63.0%  100.0%   37.0%  NORM
unexpected                      3568          4   30.3%    0.0%   30.2%  ANOM
error              

## 4. Fingerprint with Discovered Tokens

Use the auto-discovered discriminative tokens (not hand-picked) to fingerprint each session.
Binary fingerprint: which discriminative tokens are present?

In [4]:
# Only use tokens that are more common in anomalies (direction=ANOM)
# Tokens more common in normals indicate "missing normal operations", not error indicators
anom_discriminative = [w for w, ar, nr, d in hdfs_features if d == 'ANOM']
norm_discriminative = [w for w, ar, nr, d in hdfs_features if d == 'NORM']

print(f'Anomaly-enriched tokens ({len(anom_discriminative)}): {anom_discriminative}')
print(f'Normal-enriched tokens  ({len(norm_discriminative)}): {norm_discriminative}')
print(f'(Normal-enriched = operations MISSING in anomalies → structural absence signal)')

# Fingerprint function using discovered tokens
def hdfs_fingerprint_auto(session, features):
    """Binary fingerprint: which discovered tokens are present?"""
    text = ' '.join(session.lines).lower()
    return {f: (1 if f in text else 0) for f in features}

# Use ALL discriminative tokens (both anomaly-enriched and normal-enriched)
# because "missing a normal op" is itself a signal
all_disc_tokens = [w for w, ar, nr, d in hdfs_features]

# Fingerprint all sessions
anom_fps = [(s, hdfs_fingerprint_auto(s, all_disc_tokens)) for s in hdfs_anomalies]
norm_fps = [(s, hdfs_fingerprint_auto(s, all_disc_tokens)) for s in norm_sample]

# Show average fingerprint
print(f'\n{"Token":25s} {"Anom%":>7s} {"Norm%":>7s} {"Signal":>8s}')
print('-' * 50)
for tok in all_disc_tokens:
    a_pct = sum(fp[tok] for _, fp in anom_fps) / len(anom_fps)
    n_pct = sum(fp[tok] for _, fp in norm_fps) / len(norm_fps)
    sig = 'ANOM' if tok in anom_discriminative else 'MISSING'
    print(f'{tok:25s} {a_pct:7.1%} {n_pct:7.1%} {sig:>8s}')

Anomaly-enriched tokens (28): ['not', 'unexpected', 'error', 'found', 'trying', 'blockinfo', 'volumemap', 'warn', 'transfer', 'replicate', 'starting', 'thread', 'ask', 'datatransfer', 'transmitted', 'java', 'writeblock', 'ioexception', 'read', 'stream', 'could', 'exception', 'request', 'any', 'belong', 'does', 'but', 'redundant']
Normal-enriched tokens  (20): ['blockmap', 'addstoredblock', 'size', 'updated', 'added', 'packetresponder', 'for', 'terminating', 'delete', 'invalidset', 'hadoop', 'deleting', 'mnt', 'current', 'data', 'fsdataset', 'file', 'subdir', 'from', 'received']
(Normal-enriched = operations MISSING in anomalies → structural absence signal)

Token                       Anom%   Norm%   Signal
--------------------------------------------------
not                         56.9%    0.0%     ANOM
blockmap                    62.9%  100.0%  MISSING
addstoredblock              62.9%  100.0%  MISSING
size                        62.9%  100.0%  MISSING
updated                     

## 5. Cluster by Discovered Fingerprints

Group anomalies by their binary fingerprint over discovered tokens.

In [5]:
def fingerprint_label(fp):
    """Human-readable label: list tokens that are present."""
    present = sorted(k for k, v in fp.items() if v == 1)
    return '+'.join(present) if present else 'EMPTY'

# Group anomalies by their binary fingerprint
clusters = defaultdict(list)
for s, fp in anom_fps:
    key = fingerprint_label(fp)
    clusters[key].append((s, fp))

sorted_clusters = sorted(clusters.items(), key=lambda x: -len(x[1]))

print(f'Discovered {len(sorted_clusters)} unique anomaly fingerprints (raw)')
print(f'{"#":>3s} {"Fingerprint":70s} {"Count":>7s} {"% Total":>8s}')
print('-' * 90)
for i, (label, members) in enumerate(sorted_clusters, 1):
    pct = len(members) / len(hdfs_anomalies)
    display = label[:68] + '..' if len(label) > 70 else label
    print(f'{i:3d} {display:70s} {len(members):7d} {pct:8.1%}')

covered = sum(len(m) for _, m in sorted_clusters)
print(f'\nTotal: {covered} / {len(hdfs_anomalies)} (100%)')

Discovered 68 unique anomaly fingerprints (raw)
  # Fingerprint                                                              Count  % Total
------------------------------------------------------------------------------------------
  1 added+addstoredblock+ask+blockinfo+blockmap+current+data+delete+dele..    2399    20.4%
  2 ask+could+data+exception+from+ioexception+java+not+read+received+str..    2260    19.2%
  3 ask+data                                                                  2091    17.7%
  4 added+addstoredblock+ask+blockmap+current+data+datatransfer+delete+d..    1068     9.1%
  5 added+addstoredblock+ask+blockmap+current+data+datatransfer+delete+d..     773     6.6%
  6 added+addstoredblock+ask+blockinfo+blockmap+current+data+delete+dele..     634     5.4%
  7 added+addstoredblock+any+ask+belong+blockmap+but+current+data+delete..     557     4.7%
  8 added+addstoredblock+ask+blockinfo+blockmap+current+data+datatransfe..     394     3.3%
  9 added+addstoredblock+ask+bloc

## 6. Merge Overlapping Clusters

Raw fingerprints produce too many fine-grained clusters that differ by only ±1 token.
Merge clusters that share the same **core anomaly-enriched tokens** (the operations that make them anomalous).

Strategy: Two fingerprints merge if their anomaly-enriched tokens are identical
(they only differ in normal-enriched tokens, which indicate "how much of the normal pipeline completed").

In [6]:
# Merge key: only the anomaly-enriched tokens that are present
def merge_key(fp):
    """Merge key = only the anomaly-enriched tokens present in this session."""
    return tuple(sorted(k for k, v in fp.items() if v == 1 and k in anom_discriminative))

def merge_label(mk):
    return '+'.join(mk) if mk else 'NO_ANOM_TOKENS'

# Merge clusters
merged = defaultdict(list)
for s, fp in anom_fps:
    mk = merge_key(fp)
    merged[merge_label(mk)].append((s, fp))

sorted_merged = sorted(merged.items(), key=lambda x: -len(x[1]))

print(f'After merging: {len(sorted_merged)} clusters (was {len(sorted_clusters)} raw)')
print(f'{"#":>3s} {"Core anomaly tokens":55s} {"Count":>7s} {"% Total":>8s} {"Cum%":>7s}')
print('-' * 82)
cum = 0
for i, (label, members) in enumerate(sorted_merged, 1):
    n = len(members)
    pct = n / len(hdfs_anomalies)
    cum += pct
    display = label[:53] + '..' if len(label) > 55 else label
    print(f'{i:3d} {display:55s} {n:7d} {pct:8.1%} {cum:7.1%}')

total_merged = sum(len(m) for _, m in sorted_merged)
print(f'\nTotal: {total_merged} / {len(hdfs_anomalies)} (100%)')

# Also show what normal-enriched tokens are typically missing per cluster
print(f'\n=== What normal ops are missing in each cluster? ===')
for label, members in sorted_merged[:10]:
    # For each cluster, count how often each normal-enriched token is ABSENT
    absent_rates = {}
    for tok in norm_discriminative:
        absent = sum(1 for _, fp in members if fp.get(tok, 0) == 0)
        absent_rates[tok] = absent / len(members)
    missing = [f'{t}({r:.0%})' for t, r in absent_rates.items() if r > 0.3]
    print(f'  {label[:50]:50s}  missing: {", ".join(missing) if missing else "none"}')

After merging: 45 clusters (was 68 raw)
  # Core anomaly tokens                                       Count  % Total    Cum%
----------------------------------------------------------------------------------
  1 ask+blockinfo+error+found+not+trying+unexpected+volum..    2477    21.0%   21.0%
  2 ask                                                        2425    20.6%   41.6%
  3 ask+could+exception+ioexception+java+not+read+stream+..    2260    19.2%   60.8%
  4 ask+datatransfer+read+replicate+starting+thread+trans..    1096     9.3%   70.1%
  5 ask+datatransfer+exception+read+replicate+starting+th..     819     6.9%   77.0%
  6 ask+blockinfo+error+exception+found+not+trying+unexpe..     666     5.7%   82.7%
  7 any+ask+belong+but+does+exception+not+request+warn          574     4.9%   87.5%
  8 ask+redundant+request+warn                                  511     4.3%   91.9%
  9 ask+blockinfo+datatransfer+error+found+not+read+repli..     399     3.4%   95.3%
 10 any+ask+belong+but+does

## 7. Inspect Merged Clusters

Show example sessions from each merged cluster.

In [16]:
# Show 1 example from each merged cluster
for i, (label, members) in enumerate(sorted_merged[:10], 1):
    s, fp = members[0]
    present_anom = [k for k in anom_discriminative if fp.get(k, 0) == 1]
    missing_norm = [k for k in norm_discriminative if fp.get(k, 0) == 0]
    
    print(f'=== Merged Cluster #{i}: {label} ({len(members)} sessions) ===')
    print(f'  Anomaly tokens present:  {present_anom}')
    print(f'  Normal tokens missing:   {missing_norm}')
    print(f'  Session: {s.session_id} ({len(s.lines)} lines)')
    for line in s.lines[:4]:
        print(f'    {line[:120]}')
    if len(s.lines) > 4:
        print(f'    ... ({len(s.lines) - 4} more lines)')
    print()

=== Merged Cluster #1: ask+blockinfo+error+found+not+trying+unexpected+volumemap+warn (2477 sessions) ===
  Anomaly tokens present:  ['not', 'error', 'found', 'trying', 'unexpected', 'volumemap', 'blockinfo', 'warn', 'ask']
  Normal tokens missing:   []
  Session: HDFS_blk_6161225427189509238 (20 lines)
    081111 065922 22551 INFO dfs.DataNode$DataXceiver: Receiving block blk_6161225427189509238 src: /10.251.111.80:52524 des
    081111 065922 22649 INFO dfs.DataNode$DataXceiver: Receiving block blk_6161225427189509238 src: /10.251.111.80:55856 des
    081111 065922 34 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /user/root/rand6/_temporary/_task_200811101024_
    081111 065923 22695 INFO dfs.DataNode$DataXceiver: Receiving block blk_6161225427189509238 src: /10.251.38.214:34981 des
    ... (16 more lines)

=== Merged Cluster #2: ask (2425 sessions) ===
  Anomaly tokens present:  ['ask']
  Normal tokens missing:   ['size', 'updated', 'blockmap', 'added', 'addstoredblock', 'p

## 8. Generate HDFS Signature Cards from Merged Clusters

Each merged cluster with ≥ 3 sessions becomes a signature card.
Names follow the **COMPONENT__ERROR_TYPE** convention (double underscore separator):
- **Component** extracted from actual `dfs.*` log classes (DATANODE, NAMENODE, FSDATASET, BLOCKSCANNER)
- **Error type** from discriminative keywords mapped to HDFS operations

Severity is omitted — all sessions here are confirmed anomalies.

In [16]:
def extract_hdfs_component_severity(members, max_sample=200):
    """Analyze actual log lines from cluster members to find dominant component + severity.

    Returns:
        component: DATANODE | NAMENODE | FSDATASET | BLOCKSCANNER
        severity:  WARN | ERROR | FATAL
        error_desc: short error description from log content
    """
    # Count dfs.* classes and log levels across member sessions
    class_counts = Counter()
    level_counts = Counter()
    operation_counts = Counter()

    for s, fp in members[:max_sample]:
        text = '\n'.join(s.lines)
        # Extract component classes: dfs.DataNode$DataXceiver, dfs.FSNamesystem, etc.
        for cls in re.findall(r'dfs\.(\w+)', text):
            class_counts[cls] += 1
        # Extract log severity levels
        for lvl in re.findall(r'\b(INFO|WARN|WARNING|ERROR|FATAL)\b', text):
            level_counts[lvl.replace('WARNING', 'WARN')] += 1
        # Extract operation keywords
        text_lower = text.lower()
        for op in ['writeblock', 'exception', 'received block', 'receiving block',
                    'replicate', 'replication', 'addstoredblock', 'serving',
                    'transfer', 'delete', 'deleting', 'packetresponder',
                    'dataxceiver', 'datatransfer', 'blockscanner', 'volumemap',
                    'ioexception', 'unexpected', 'timed out', 'error']:
            if op in text_lower:
                operation_counts[op] += 1

    # --- Determine component from dominant class ---
    # Map dfs classes to components
    component_map = {
        'DataNode': 'DATANODE',
        'FSNamesystem': 'NAMENODE',
        'FSDataset': 'FSDATASET',
        'DataBlockScanner': 'BLOCKSCANNER',
        'PendingReplicationBlocks': 'NAMENODE',
    }
    component = 'DATANODE'  # default
    for cls, _ in class_counts.most_common():
        # Strip inner class: DataNode$DataXceiver → DataNode
        base_cls = cls.split('$')[0] if '$' in cls else cls
        if base_cls in component_map:
            component = component_map[base_cls]
            break

    # --- Determine severity ---
    # For anomalies, prefer the highest severity seen, not the most common
    # (most lines are INFO but the anomaly signal is in WARN/ERROR/FATAL)
    severity = 'WARN'  # default sensible for anomalies
    for lvl in ['FATAL', 'ERROR', 'WARN']:
        if level_counts.get(lvl, 0) > 0:
            severity = lvl
            break

    return component, severity, operation_counts


def auto_name_hdfs(label, members):
    """Generate COMPONENT__ERROR_TYPE signature from actual log content."""
    tokens = set(label.split('+')) if label != 'NO_ANOM_TOKENS' else set()
    component, _severity, ops = extract_hdfs_component_severity(members)

    # --- Determine error type from cluster's discriminative tokens + log ops ---
    n = len(members)

    # Priority-ordered error type mapping
    if 'writeblock' in tokens and 'exception' in tokens:
        error_type = 'WRITE_PIPELINE_EXCEPTION'
        desc = f'Write pipeline failures with exceptions ({n} sessions)'
    elif 'writeblock' in tokens:
        error_type = 'BLOCK_WRITE_FAILURE'
        desc = f'Block write operations failed to complete ({n} sessions)'
    elif 'exception' in tokens and 'replicate' in tokens:
        error_type = 'REPLICATION_EXCEPTION'
        desc = f'Replication with exceptions detected ({n} sessions)'
    elif 'replicate' in tokens:
        error_type = 'REPLICATION_INCOMPLETE'
        component = 'NAMENODE'  # replication is managed by NameNode
        desc = f'Block replication incomplete or triggered re-replication ({n} sessions)'
    elif 'redundant' in tokens and 'request' in tokens:
        error_type = 'REDUNDANT_STORED_BLOCK'
        component = 'NAMENODE'
        desc = f'Redundant addStoredBlock requests for already-known blocks ({n} sessions)'
    elif 'serving' in tokens:
        error_type = 'BLOCK_SERVING_FAILURE'
        desc = f'Block serving errors during read requests ({n} sessions)'
    elif 'exception' in tokens and ('volumemap' in tokens or 'unexpected' in tokens):
        error_type = 'BLOCK_VERIFICATION_FAILED'
        desc = f'Block verification/volume map errors with exceptions ({n} sessions)'
    elif 'exception' in tokens:
        error_type = 'PACKET_RESPONDER_EXCEPTION'
        desc = f'Exception in DataNode packet handling ({n} sessions)'
    elif 'error' in tokens and ('volumemap' in tokens or 'blockinfo' in tokens):
        error_type = 'BLOCK_VERIFICATION_FAILED'
        desc = f'Block info or volume map errors detected ({n} sessions)'
    elif 'error' in tokens:
        error_type = 'BLOCK_OPERATION_ERROR'
        desc = f'Block operation errors detected ({n} sessions)'
    elif 'belong' in tokens or ('does' in tokens and 'not' in tokens):
        # "does not belong to any file" = orphan block on NameNode
        error_type = 'ORPHAN_BLOCK'
        component = 'NAMENODE'
        desc = f'addStoredBlock for blocks not belonging to any file ({n} sessions)'
    elif 'ask' in tokens and not tokens - {'ask'}:
        # "ask" alone = incomplete pipeline (receiving without completing)
        error_type = 'BLOCK_RECEIVE_INCOMPLETE'
        desc = f'Block receive started but pipeline incomplete ({n} sessions)'
    elif 'warn' in tokens and not (tokens - {'warn', 'ask', 'not'}):
        error_type = 'BLOCK_OPERATION_WARNING'
        desc = f'Block operations with warnings ({n} sessions)'
    elif not tokens:
        # Structural anomaly — normal ops missing
        error_type = 'STRUCTURAL_ANOMALY'
        desc = f'No error keywords but normal operations missing ({n} sessions)'
    else:
        # Fallback: derive from actual log component analysis
        error_type = 'BLOCK_OPERATION_ERROR'
        desc = f'Block operation anomaly with tokens: {", ".join(sorted(tokens)[:5])} ({n} sessions)'

    name = f'{component}__{error_type}'

    return name, desc


# Generate patterns from merged clusters
print('=== Auto-Generated HDFS Signature Cards (from merged clusters) ===\n')
hdfs_patterns_discovered = {}
pattern_id = 0

for label, members in sorted_merged:
    if len(members) < 3:
        continue

    pattern_id += 1
    name, desc = auto_name_hdfs(label, members)

    # Keywords = the anomaly-enriched tokens in this cluster
    tokens = [t for t in label.split('+') if t and t != 'NO_ANOM_TOKENS']

    # Normal-enriched tokens typically missing
    absent_norms = []
    for tok in norm_discriminative:
        absent = sum(1 for _, fp in members if fp.get(tok, 0) == 0)
        if absent / len(members) > 0.5:
            absent_norms.append(tok)

    key = f'hdfs_{pattern_id:02d}'
    hdfs_patterns_discovered[key] = {
        'name': name,
        'description': desc,
        'keywords': tokens,
        'patterns': [re.escape(t) for t in tokens],
        'frequency': len(members),
        'merge_key': label,
        'typically_missing': absent_norms,
    }

    print(f'{pattern_id:2d}. {name}')
    print(f'    Description: {desc}')
    print(f'    Core tokens: {tokens}')
    print(f'    Sessions: {len(members)}')
    print(f'    Usually missing: {absent_norms}')
    print()

total_covered = sum(p['frequency'] for p in hdfs_patterns_discovered.values())
print(f'Patterns: {len(hdfs_patterns_discovered)}')
print(f'Coverage: {total_covered}/{len(hdfs_anomalies)} ({total_covered/len(hdfs_anomalies):.1%})')
uncovered = len(hdfs_anomalies) - total_covered
print(f'Uncovered (< 3 sessions): {uncovered} ({uncovered/len(hdfs_anomalies):.1%})')

=== Auto-Generated HDFS Signature Cards (from merged clusters) ===

 1. DATANODE__BLOCK_VERIFICATION_FAILED
    Description: Block info or volume map errors detected (2477 sessions)
    Core tokens: ['ask', 'blockinfo', 'error', 'found', 'not', 'trying', 'unexpected', 'volumemap', 'warn']
    Sessions: 2477
    Usually missing: []

 2. DATANODE__BLOCK_RECEIVE_INCOMPLETE
    Description: Block receive started but pipeline incomplete (2425 sessions)
    Core tokens: ['ask']
    Sessions: 2425
    Usually missing: ['blockmap', 'addstoredblock', 'size', 'updated', 'added', 'packetresponder', 'for', 'terminating', 'delete', 'invalidset', 'hadoop', 'deleting', 'mnt', 'current', 'fsdataset', 'file', 'subdir', 'from', 'received']

 3. DATANODE__WRITE_PIPELINE_EXCEPTION
    Description: Write pipeline failures with exceptions (2260 sessions)
    Core tokens: ['ask', 'could', 'exception', 'ioexception', 'java', 'not', 'read', 'stream', 'writeblock']
    Sessions: 2260
    Usually missing: ['bloc

## 9. Export: Python dict for `signature_generator.py`

Print the discovered patterns as Python code you can paste directly into the source.

In [17]:
# Print as Python code for signature_generator.py
print('# Auto-generated from training data — do not hand-edit')
print('# Discovery notebook: notebooks/05_signature_audit.ipynb')
print(f'# Source: {len(hdfs_anomalies)} training anomalies, {len(hdfs_patterns_discovered)} patterns')
print(f'# Naming convention: COMPONENT__ERROR_TYPE')
print(f'# Components: DATANODE, NAMENODE, FSDATASET, BLOCKSCANNER')
print()
print('HDFS_ERROR_PATTERNS = {')
for key, p in hdfs_patterns_discovered.items():
    print(f'    "{key}": {{')
    print(f'        "name": "{p["name"]}",')
    print(f'        "description": "{p["description"]}",')
    print(f'        "keywords": {p["keywords"]},')
    print(f'        "patterns": {p["patterns"]},')
    print(f'        # frequency={p["frequency"]}, merge_key={p["merge_key"]}')
    print(f'        # typically_missing={p["typically_missing"]}')
    print(f'    }},')
print('}')
print(f'\n# {len(hdfs_patterns_discovered)} patterns, covering {total_covered}/{len(hdfs_anomalies)} anomalies ({total_covered/len(hdfs_anomalies):.1%})')

# Also show deduplicated summary (some clusters may map to same signature)
from collections import Counter as C
sig_counts = C()
for p in hdfs_patterns_discovered.values():
    sig_counts[p['name']] += p['frequency']
print(f'\n# {len(sig_counts)} unique signature names after dedup:')
for sig, count in sig_counts.most_common():
    print(f'#   {count:5d}  {sig}')

# Auto-generated from training data — do not hand-edit
# Discovery notebook: notebooks/05_signature_audit.ipynb
# Source: 11786 training anomalies, 26 patterns
# Naming convention: COMPONENT__ERROR_TYPE
# Components: DATANODE, NAMENODE, FSDATASET, BLOCKSCANNER

HDFS_ERROR_PATTERNS = {
    "hdfs_01": {
        "name": "DATANODE__BLOCK_VERIFICATION_FAILED",
        "description": "Block info or volume map errors detected (2477 sessions)",
        "keywords": ['ask', 'blockinfo', 'error', 'found', 'not', 'trying', 'unexpected', 'volumemap', 'warn'],
        "patterns": ['ask', 'blockinfo', 'error', 'found', 'not', 'trying', 'unexpected', 'volumemap', 'warn'],
        # frequency=2477, merge_key=ask+blockinfo+error+found+not+trying+unexpected+volumemap+warn
        # typically_missing=[]
    },
    "hdfs_02": {
        "name": "DATANODE__BLOCK_RECEIVE_INCOMPLETE",
        "description": "Block receive started but pipeline incomplete (2425 sessions)",
        "keywords": ['ask'],
        "p

---

## 10. Load BGL Training Data

In [11]:
bgl_loader = BGLDataLoader(log_file='../logs/BGL.log')
bgl_loader.load()

bgl_train = bgl_loader.get_train()
bgl_anomalies = [s for s in bgl_train if s.label == 1]
bgl_normals   = [s for s in bgl_train if s.label == 0]

print(f'BGL Training sessions: {len(bgl_train):,}')
print(f'  Anomalies: {len(bgl_anomalies):,}')
print(f'  Normals:   {len(bgl_normals):,}')

Loading BGL logs from: ../logs/BGL.log


Reading BGL logs: 4747963it [00:01, 3847841.58it/s]


Loaded 4747963 log lines


Creating sessions: 100%|██████████| 474796/474796 [00:02<00:00, 164356.14it/s]


BGL Training sessions: 332,356
  Anomalies: 27,315
  Normals:   305,041


## 11. BGL Keyword Discovery

BGL anomalies are keyword-rich (FATAL, machine check, parity error, etc.).
Extract the most discriminative keywords directly from the data.

In [12]:
# Extract significant tokens from BGL anomaly lines
# BGL lines have rich error keywords unlike HDFS

# Tokenize and count: which words appear much more in anomalies than normals?
anom_word_counts = Counter()
norm_word_counts = Counter()

for s in bgl_anomalies:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'\b[a-z]{3,}\b', text))  # unique words per session
    for w in words:
        anom_word_counts[w] += 1

random.seed(42)
bgl_norm_sample = random.sample(bgl_normals, min(len(bgl_anomalies), len(bgl_normals)))
for s in bgl_norm_sample:
    text = ' '.join(s.lines).lower()
    words = set(re.findall(r'\b[a-z]{3,}\b', text))
    for w in words:
        norm_word_counts[w] += 1

# Compute discriminative score: (anom_rate - norm_rate) * anom_count
# Only consider words appearing in at least 5 anomaly sessions
print(f'{"Word":25s} {"Anom sessions":>14s} {"Norm sessions":>14s} {"Anom %":>8s} {"Norm %":>8s} {"Disc.":>8s}')
print('-' * 75)

disc_scores = []
for word, a_count in anom_word_counts.items():
    if a_count < 5:
        continue
    n_count = norm_word_counts.get(word, 0)
    a_rate = a_count / len(bgl_anomalies)
    n_rate = n_count / max(len(bgl_norm_sample), 1)
    disc = (a_rate - n_rate) * a_count  # high when anomaly-specific and frequent
    disc_scores.append((word, a_count, n_count, a_rate, n_rate, disc))

disc_scores.sort(key=lambda x: -x[5])
for word, a_count, n_count, a_rate, n_rate, disc in disc_scores[:40]:
    print(f'{word:25s} {a_count:14d} {n_count:14d} {a_rate:8.1%} {n_rate:8.1%} {disc:8.1f}')

Word                       Anom sessions  Norm sessions   Anom %   Norm %    Disc.
---------------------------------------------------------------------------
fatal                              27294           3178    99.9%    11.6%  24097.5
data                               16357            472    59.9%     1.7%   9512.4
interrupt                          16627           2899    60.9%    10.6%   8356.4
error                              17271           4093    63.2%    15.0%   8332.3
tlb                                11075             12    40.5%     0.0%   4485.5
message                             5658             68    20.7%     0.2%   1157.9
socket                              5277              1    19.3%     0.0%   1019.3
prefix                              5244              0    19.2%     0.0%   1006.8
ciostream                           5244              0    19.2%     0.0%   1006.8
storage                             5199              0    19.0%     0.0%    989.6
app        

## 12. Discover & Merge BGL Anomaly Clusters

Same approach as HDFS: use the discriminative words from step 11, fingerprint, cluster, and merge.
For BGL, anomaly tokens are the merge keys (since BGL is already keyword-rich).

In [13]:
# Auto-select discriminative features for BGL
# Keep words that are anomaly-enriched: anom_rate > norm_rate and difference > 5%
bgl_disc = [(w, ac, nc, ar, nr) for w, ac, nc, ar, nr, d in disc_scores 
            if ar - nr > 0.05 and ar > 0.02]
bgl_features = [w for w, ac, nc, ar, nr in bgl_disc]

print(f'Selected {len(bgl_features)} discriminative BGL features:')
for w, ac, nc, ar, nr in bgl_disc[:20]:
    print(f'  {w:25s}  anom={ar:.1%}  norm={nr:.1%}  diff={ar-nr:.1%}')
if len(bgl_disc) > 20:
    print(f'  ... and {len(bgl_disc) - 20} more')

# Fingerprint BGL anomalies
def bgl_fingerprint(session, features):
    text = ' '.join(session.lines).lower()
    return {f: (1 if f in text else 0) for f in features}

bgl_fps = [(s, bgl_fingerprint(s, bgl_features)) for s in bgl_anomalies]

# Cluster by full fingerprint
bgl_clusters = defaultdict(list)
for s, fp in bgl_fps:
    key = '+'.join(sorted(k for k, v in fp.items() if v == 1)) or 'NONE'
    bgl_clusters[key].append((s, fp))

bgl_sorted_raw = sorted(bgl_clusters.items(), key=lambda x: -len(x[1]))
print(f'\nRaw BGL clusters: {len(bgl_sorted_raw)}')

# For BGL, merging is less critical (keywords are already specific)
# But let's still merge by a "primary error type" heuristic:
# Use only the top-3 most discriminative tokens present as the merge key
# (since BGL fingerprints can have many tokens from overlapping error messages)
top_bgl_tokens = set(bgl_features[:15])  # most discriminative

def bgl_merge_key(fp):
    """Merge key = top discriminative tokens present."""
    present = sorted(k for k, v in fp.items() if v == 1 and k in top_bgl_tokens)
    return '+'.join(present) if present else 'OTHER'

bgl_merged = defaultdict(list)
for s, fp in bgl_fps:
    mk = bgl_merge_key(fp)
    bgl_merged[mk].append((s, fp))

bgl_sorted = sorted(bgl_merged.items(), key=lambda x: -len(x[1]))

print(f'Merged BGL clusters: {len(bgl_sorted)} (from {len(bgl_sorted_raw)} raw)')
print(f'\n{"#":>3s} {"Merge key":55s} {"Count":>7s} {"% Total":>8s} {"Cum%":>7s}')
print('-' * 82)
cum = 0
for i, (label, members) in enumerate(bgl_sorted[:20], 1):
    n = len(members)
    pct = n / len(bgl_anomalies)
    cum += pct
    display = label[:53] + '..' if len(label) > 55 else label
    print(f'{i:3d} {display:55s} {n:7d} {pct:8.1%} {cum:7.1%}')

Selected 25 discriminative BGL features:
  fatal                      anom=99.9%  norm=11.6%  diff=88.3%
  data                       anom=59.9%  norm=1.7%  diff=58.2%
  interrupt                  anom=60.9%  norm=10.6%  diff=50.3%
  error                      anom=63.2%  norm=15.0%  diff=48.2%
  tlb                        anom=40.5%  norm=0.0%  diff=40.5%
  message                    anom=20.7%  norm=0.2%  diff=20.5%
  socket                     anom=19.3%  norm=0.0%  diff=19.3%
  prefix                     anom=19.2%  norm=0.0%  diff=19.2%
  ciostream                  anom=19.2%  norm=0.0%  diff=19.2%
  storage                    anom=19.0%  norm=0.0%  diff=19.0%
  app                        anom=20.7%  norm=3.9%  diff=16.8%
  reading                    anom=17.7%  norm=0.0%  diff=17.7%
  ciod                       anom=20.8%  norm=6.6%  diff=14.1%
  been                       anom=16.7%  norm=0.0%  diff=16.7%
  has                        anom=16.7%  norm=0.1%  diff=16.7%
  link     

## 13. Inspect Top BGL Clusters

In [22]:
# Show 1 example from each top BGL merged cluster
for i, (label, members) in enumerate(bgl_sorted[:10], 1):
    s = members[0][0]
    print(f'=== BGL Cluster #{i}: {label} ({len(members)} sessions) ===')
    for line in s.lines[:3]:
        print(f'    {line[:130]}')
    print()

=== BGL Cluster #1: data+error+fatal+interrupt+tlb (11052 sessions) ===
    1118539262 2005.06.11 R30-M0-N9-C:J16-U01 2005-06-11-18.21.02.351205 R30-M0-N9-C:J16-U01 RAS KERNEL FATAL data TLB error interrupt
    1118539262 2005.06.11 R30-M0-N9-C:J16-U01 2005-06-11-18.21.02.456536 R30-M0-N9-C:J16-U01 RAS KERNEL FATAL data TLB error interrupt
    1118539262 2005.06.11 R30-M0-N9-C:J16-U01 2005-06-11-18.21.02.596755 R30-M0-N9-C:J16-U01 RAS KERNEL FATAL data TLB error interrupt

=== BGL Cluster #2: data+fatal+interrupt+storage (5196 sessions) ===
    1118769401 2005.06.14 R27-M0-ND-C:J03-U01 2005-06-14-10.16.41.287198 R27-M0-ND-C:J03-U01 RAS KERNEL FATAL instruction address: 0x0
    1118769401 2005.06.14 R25-M1-ND-C:J03-U01 2005-06-14-10.16.41.294988 R25-M1-ND-C:J03-U01 RAS KERNEL FATAL data storage interrupt
    1118769401 2005.06.14 R27-M0-ND-C:J05-U11 2005-06-14-10.16.41.309072 R27-M0-ND-C:J05-U11 RAS KERNEL FATAL instruction address: 0x0

=== BGL Cluster #3: app+been+ciod+ciostream+error

## 14. Generate BGL Signature Cards (COMPONENT__ERROR_TYPE)

In [19]:
# Generate BGL patterns from merged clusters (>= 3 sessions)
# Name as COMPONENT__ERROR_TYPE from actual BGL log content

def extract_bgl_component(members, max_sample=200):
    """Extract dominant RAS component (KERNEL, APP, MMCS, LINKCARD) from BGL sessions."""
    comp_counts = Counter()
    for s, fp in members[:max_sample]:
        text = ' '.join(s.lines)
        for comp in re.findall(r'RAS\s+(KERNEL|APP|MMCS|LINKCARD|BGLMASTER)', text):
            comp_counts[comp] += 1
    return comp_counts.most_common(1)[0][0] if comp_counts else 'KERNEL'


def auto_name_bgl(label, members):
    """Generate COMPONENT__ERROR_TYPE signature from BGL cluster keywords + log content."""
    tokens = set(label.split('+')) if label else set()
    component = extract_bgl_component(members)
    n = len(members)

    # Priority-ordered error type mapping based on BGL log message patterns
    if 'tlb' in tokens and 'data' in tokens:
        error_type = 'DATA_TLB_ERROR'
        desc = f'Data TLB (translation lookaside buffer) error interrupt ({n} sessions)'
    elif 'storage' in tokens and 'data' in tokens and 'interrupt' in tokens:
        error_type = 'DATA_STORAGE_INTERRUPT'
        desc = f'Data storage interrupt — memory/bus fault ({n} sessions)'
    elif 'machine' in tokens and 'check' in tokens:
        error_type = 'MACHINE_CHECK'
        desc = f'Machine check exception — hardware fault detected ({n} sessions)'
    elif 'parity' in tokens:
        error_type = 'CACHE_PARITY_ERROR'
        desc = f'Cache parity error in instruction/data cache ({n} sessions)'
    elif 'ciostream' in tokens and 'socket' in tokens:
        error_type = 'CIOD_SOCKET_ERROR'
        desc = f'CIOD stream socket communication failure ({n} sessions)'
    elif 'ciostream' in tokens and 'message' in tokens:
        error_type = 'CIOD_MESSAGE_ERROR'
        desc = f'CIOD error reading/writing message on CioStream ({n} sessions)'
    elif 'ciostream' in tokens:
        error_type = 'CIOD_STREAM_ERROR'
        desc = f'CIOD CioStream I/O error ({n} sessions)'
    elif 'ciod' in tokens and 'socket' in tokens:
        error_type = 'CIOD_SOCKET_ERROR'
        desc = f'CIOD socket error ({n} sessions)'
    elif 'ciod' in tokens and 'message' in tokens:
        error_type = 'CIOD_MESSAGE_ERROR'
        desc = f'CIOD message prefix read/write error ({n} sessions)'
    elif 'ciod' in tokens and 'been' in tokens:
        error_type = 'CIOD_SIGNAL_DELIVERED'
        desc = f'CIOD signal delivered — process killed or interrupted ({n} sessions)'
    elif 'ciod' in tokens:
        error_type = 'CIOD_ERROR'
        desc = f'CIOD (compute I/O daemon) error ({n} sessions)'
        component = 'APP'  # ciod errors come from APP subsystem
    elif 'interrupt' in tokens and 'data' in tokens:
        error_type = 'DATA_INTERRUPT'
        desc = f'Data interrupt — memory or bus error ({n} sessions)'
    elif 'interrupt' in tokens:
        error_type = 'PROGRAM_INTERRUPT'
        desc = f'Program interrupt exception ({n} sessions)'
    elif 'message' in tokens and 'fatal' in tokens:
        error_type = 'FATAL_MESSAGE'
        desc = f'Fatal message — unrecoverable error reported ({n} sessions)'
    elif 'data' in tokens and 'error' in tokens:
        error_type = 'DATA_ERROR'
        desc = f'Data error — correctable or uncorrectable memory fault ({n} sessions)'
    elif 'error' in tokens:
        error_type = 'HARDWARE_ERROR'
        desc = f'Hardware error detected ({n} sessions)'
    elif 'fatal' in tokens:
        error_type = 'FATAL_ERROR'
        desc = f'Fatal error — unrecoverable fault ({n} sessions)'
    else:
        error_type = 'UNKNOWN_ERROR'
        desc = f'Anomaly cluster with tokens: {label} ({n} sessions)'

    name = f'{component}__{error_type}'
    return name, desc


bgl_patterns_discovered = {}
pid = 0

for label, members in bgl_sorted:
    if len(members) < 3:
        continue
    pid += 1

    keywords = [k for k in label.split('+') if k]
    name, desc = auto_name_bgl(label, members)

    key = f'bgl_auto_{pid:02d}'
    bgl_patterns_discovered[key] = {
        'name': name,
        'description': desc,
        'keywords': keywords,
        'patterns': [r'\b' + re.escape(k) + r'\b' if len(k) > 3 else k for k in keywords[:5]],
        'frequency': len(members),
        'fingerprint': label,
    }

    print(f'{pid:2d}. {name}')
    print(f'    Description: {desc}')
    print(f'    Keywords: {keywords[:5]}')
    print()

total_bgl = sum(p['frequency'] for p in bgl_patterns_discovered.values())
print(f'{len(bgl_patterns_discovered)} patterns, covering {total_bgl}/{len(bgl_anomalies)} ({total_bgl/len(bgl_anomalies):.1%})')

# Show deduplicated summary
from collections import Counter as C
bgl_sig_counts = C()
for p in bgl_patterns_discovered.values():
    bgl_sig_counts[p['name']] += p['frequency']
print(f'\n{len(bgl_sig_counts)} unique BGL signature names:')
for sig, count in bgl_sig_counts.most_common():
    print(f'  {count:5d}  {sig}')

 1. KERNEL__DATA_TLB_ERROR
    Description: Data TLB (translation lookaside buffer) error interrupt (11052 sessions)
    Keywords: ['data', 'error', 'fatal', 'interrupt', 'tlb']

 2. KERNEL__DATA_STORAGE_INTERRUPT
    Description: Data storage interrupt — memory/bus fault (5196 sessions)
    Keywords: ['data', 'fatal', 'interrupt', 'storage']

 3. APP__CIOD_SOCKET_ERROR
    Description: CIOD stream socket communication failure (4450 sessions)
    Keywords: ['app', 'been', 'ciod', 'ciostream', 'error']

 4. KERNEL__FATAL_ERROR
    Description: Fatal error — unrecoverable fault (3941 sessions)
    Keywords: ['fatal']

 5. KERNEL__HARDWARE_ERROR
    Description: Hardware error detected (668 sessions)
    Keywords: ['error', 'fatal']

 6. APP__CIOD_SOCKET_ERROR
    Description: CIOD stream socket communication failure (399 sessions)
    Keywords: ['app', 'ciod', 'ciostream', 'fatal', 'message']

 7. APP__CIOD_ERROR
    Description: CIOD (compute I/O daemon) error (388 sessions)
    Keyword

## 15. Summary

**Fully data-driven approach** — no hand-crafted patterns:
1. **Discover** discriminative tokens by comparing anomaly vs normal word frequencies
2. **Fingerprint** every anomaly session using only discovered tokens
3. **Cluster** by fingerprint, then **merge** overlapping clusters (anomaly-enriched tokens as merge key)
4. **Auto-name** each cluster from its dominant keywords
5. **Export** as Python dicts ready for `signature_generator.py`

**Next steps** after reviewing the output:
- Copy the generated `HDFS_ERROR_PATTERNS` and `BGL_ERROR_PATTERNS` into `src/signature_generator.py`
- All patterns are 100% grounded in training data — zero guessing

In [20]:
# Save patterns to patterns/ directory (permanent, no need to rerun discovery)
import json
from pathlib import Path

patterns_dir = Path("..") / "patterns"
patterns_dir.mkdir(exist_ok=True)

with open(patterns_dir / "hdfs_patterns.json", "w") as f:
    json.dump(hdfs_patterns_discovered, f, indent=2)

with open(patterns_dir / "bgl_patterns.json", "w") as f:
    json.dump(bgl_patterns_discovered, f, indent=2)

print(f"Saved {len(hdfs_patterns_discovered)} HDFS patterns -> {patterns_dir.resolve() / 'hdfs_patterns.json'}")
print(f"Saved {len(bgl_patterns_discovered)} BGL patterns  -> {patterns_dir.resolve() / 'bgl_patterns.json'}")

Saved 26 HDFS patterns -> /home/dave/agentic-log-explanations/patterns/hdfs_patterns.json
Saved 34 BGL patterns  -> /home/dave/agentic-log-explanations/patterns/bgl_patterns.json
